# Phase 5: Evaluate Sentence-BERT Model on Unseen Labeled Data

This notebook evaluates the trained Sentence-BERT pipeline on unseen labeled data.


In [4]:
import sys
import subprocess
from pathlib import Path

req = Path("notebooks/requirements.txt")
if not req.exists():
    req = Path("../notebooks/requirements.txt")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(req)])


0

In [5]:
from pathlib import Path
from datetime import datetime
import json
import pickle
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sentence_transformers import SentenceTransformer

def log(msg: str) -> None:
    print(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {msg}", flush=True)

PROJECT_ROOT = Path(r"D:/client-projects/sl-social-media-risk-analysis")
DATA_ROOT = Path('/root/separate_volume')
if DATA_ROOT.exists():
    ROOT = DATA_ROOT
elif PROJECT_ROOT.exists():
    ROOT = PROJECT_ROOT
else:
    ROOT = Path.cwd().resolve()
    for p in [ROOT] + list(ROOT.parents):
        if (p / "data").exists() and (p / "annotation").exists() and (p / "training").exists():
            ROOT = p
            break

RUNS_ROOT = ROOT / "training/artifacts/runs"
LATEST_RUN_FILE = RUNS_ROOT / "latest_run.txt"

if LATEST_RUN_FILE.exists():
    RUN_ROOT = Path(LATEST_RUN_FILE.read_text(encoding="utf-8").strip())
    MODEL_ROOT = RUN_ROOT / "model"
    REPORT_DIR = RUN_ROOT / "reports"
else:
    # Backward compatibility with old fixed-path layout.
    MODEL_ROOT = ROOT / "training/artifacts/models_phase5_sbert"
    REPORT_DIR = ROOT / "training/artifacts/reports_phase5_sbert"

SBERT_DIR = MODEL_ROOT / "sbert_model"
CLF_PATH = MODEL_ROOT / "embedding_classifier.pkl"
UNSEEN_PATH = ROOT / "datasets/splits/current/unseen_labeled_rest.csv"
REPORT_DIR.mkdir(parents=True, exist_ok=True)
log(f"Using model root: {MODEL_ROOT}")
log(f"Using report dir: {REPORT_DIR}")

ENCODE_BATCH_SIZE = 32
ENCODE_CHUNK_SIZE = 2000

if not SBERT_DIR.exists():
    raise FileNotFoundError(f"Missing Sentence-BERT model: {SBERT_DIR}")
if not CLF_PATH.exists():
    raise FileNotFoundError(f"Missing embedding classifier: {CLF_PATH}")
if not UNSEEN_PATH.exists():
    fallback = ROOT / "datasets/labeled/annotator_a_llm.csv"
    train_path = ROOT / "datasets/splits/current/train_labeled_331.csv"
    if fallback.exists() and train_path.exists():
        log("Unseen split missing. Rebuilding from labeled file + train IDs...")
        a = pd.read_csv(fallback)
        t = pd.read_csv(train_path)
        a["annotator_label"] = a["annotator_label"].fillna("").astype(str).str.strip().str.upper()
        labeled = a[a["annotator_label"].isin(["NORMAL", "HATE", "DISINFO"])].copy()
        train_ids = set(t["candidate_id"].astype(str))
        unseen = labeled[~labeled["candidate_id"].astype(str).isin(train_ids)].copy()
        UNSEEN_PATH.parent.mkdir(parents=True, exist_ok=True)
        unseen.to_csv(UNSEEN_PATH, index=False, encoding="utf-8")
        log(f"Created unseen split: {UNSEEN_PATH} | rows={len(unseen)}")
    else:
        raise FileNotFoundError(f"Missing unseen dataset: {UNSEEN_PATH}")

meta = json.loads((MODEL_ROOT / "meta.json").read_text(encoding="utf-8"))
label2id = {str(k): int(v) for k, v in meta["label2id"].items()}
id2label = {int(k): v for k, v in meta["id2label"].items()}
label_order = [id2label[i] for i in sorted(id2label.keys())]

log("Loading SBERT + classifier artifacts...")
model = SentenceTransformer(str(SBERT_DIR))
with CLF_PATH.open("rb") as f:
    clf = pickle.load(f)

log(f"Reading unseen dataset: {UNSEEN_PATH}")
df = pd.read_csv(UNSEEN_PATH)
text_col = "clean_text" if "clean_text" in df.columns else "text"
if "source" not in df.columns:
    df["source"] = "unknown"

df = df.copy()
df["text"] = df[text_col].fillna("").astype(str).str.replace("\u200d", "", regex=False)
df["text"] = df["text"].str.split().str.join(" ").str.strip()
df["label"] = df["annotator_label"].fillna("").astype(str).str.strip().str.upper()
df = df[(df["text"].str.len() > 0) & (df["label"].isin(label_order))].copy()
df["label_id"] = df["label"].map({k: v for k, v in label2id.items()}).astype(int)

log(f"Unseen rows ready: {len(df)}")
print(df["label"].value_counts())


2026-03-16 00:40:15 | Loading SBERT + classifier artifacts...
2026-03-16 00:40:17 | Reading unseen dataset: D:\Desktop\Projects\client\sl-social-media-risk-analysis\data\datasets\splits\current\unseen_labeled_rest.csv
2026-03-16 00:40:19 | Unseen rows ready: 57130
label
NORMAL     47451
HATE        9643
DISINFO       36
Name: count, dtype: int64


In [6]:
texts = df["text"].tolist()
total = len(texts)
embed_chunks = []
start = time.perf_counter()

log(f"Encoding start | rows={total} | chunk_size={ENCODE_CHUNK_SIZE} | batch_size={ENCODE_BATCH_SIZE}")
for start_idx in range(0, total, ENCODE_CHUNK_SIZE):
    end_idx = min(start_idx + ENCODE_CHUNK_SIZE, total)
    chunk = texts[start_idx:end_idx]
    log(f"Encoding chunk {start_idx + 1}-{end_idx}/{total}")
    emb = model.encode(
        chunk,
        batch_size=ENCODE_BATCH_SIZE,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    embed_chunks.append(emb)
    done = end_idx
    elapsed = max(time.perf_counter() - start, 1e-6)
    rate = done / elapsed
    eta = (total - done) / max(rate, 1e-6)
    log(f"Progress {done}/{total} | rate={rate:.1f} rows/s | eta={eta/60:.1f} min")

X_unseen = np.vstack(embed_chunks) if embed_chunks else np.empty((0, 0))
log(f"Encoding complete | shape={X_unseen.shape}")

y_true = df["label_id"].to_numpy()
log("Running classifier predictions...")
y_pred = clf.predict(X_unseen)
if hasattr(clf, "predict_proba"):
    y_proba = clf.predict_proba(X_unseen)
    pred_conf = y_proba.max(axis=1)
else:
    y_proba = None
    pred_conf = np.full(len(y_pred), np.nan)

acc = float(accuracy_score(y_true, y_pred))
macro_f1 = float(f1_score(y_true, y_pred, average="macro"))
cm_array = confusion_matrix(y_true, y_pred)
cm = cm_array.tolist()
report = classification_report(y_true, y_pred, target_names=label_order, output_dict=True)

pred_df = df.copy()
pred_df["y_true"] = [id2label[int(i)] for i in y_true]
pred_df["y_pred"] = [id2label[int(i)] for i in y_pred]
pred_df["pred_confidence"] = pred_conf
pred_df["is_correct"] = pred_df["y_true"] == pred_df["y_pred"]

pred_path = REPORT_DIR / "phase5_sbert_unseen_predictions.csv"
report_json_path = REPORT_DIR / "phase5_sbert_unseen_report.json"
report_csv_path = REPORT_DIR / "phase5_sbert_unseen_report.csv"
cm_json_path = REPORT_DIR / "phase5_sbert_unseen_confusion_matrix.json"
summary_path = REPORT_DIR / "phase5_sbert_unseen_summary.json"

pred_df.to_csv(pred_path, index=False, encoding="utf-8")
pd.DataFrame(report).T.to_csv(report_csv_path, encoding="utf-8")
report_json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")
cm_json_path.write_text(json.dumps({"labels": label_order, "matrix": cm}, ensure_ascii=False, indent=2), encoding="utf-8")

summary = {
    "model_root": str(MODEL_ROOT),
    "unseen_path": str(UNSEEN_PATH),
    "rows": int(len(df)),
    "metrics": {
        "accuracy": acc,
        "macro_f1": macro_f1,
        "disinfo_precision": report.get("DISINFO", {}).get("precision", None),
        "disinfo_recall": report.get("DISINFO", {}).get("recall", None),
        "disinfo_f1": report.get("DISINFO", {}).get("f1-score", None),
    },
    "artifacts": {
        "predictions_csv": str(pred_path),
        "classification_report_json": str(report_json_path),
        "classification_report_csv": str(report_csv_path),
        "confusion_matrix_json": str(cm_json_path),
    },
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

log(f"Unseen accuracy={acc:.4f} | macro_f1={macro_f1:.4f} | disinfo_recall={summary['metrics']['disinfo_recall']}")
log("Building analytics charts...")

true_counts = pred_df["y_true"].value_counts().reindex(label_order, fill_value=0)
pred_counts = pred_df["y_pred"].value_counts().reindex(label_order, fill_value=0)
count_plot_df = pd.DataFrame({"true": true_counts, "pred": pred_counts})
ax = count_plot_df.plot(kind="bar", figsize=(8, 4), title="Unseen Label Distribution (True vs Pred)")
ax.set_xlabel("Label")
ax.set_ylabel("Rows")
ax.figure.tight_layout()
ax.figure.savefig(REPORT_DIR / "phase5_unseen_label_distribution.png", dpi=140)
plt.close(ax.figure)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_array, cmap="Blues")
ax.set_xticks(range(len(label_order)))
ax.set_yticks(range(len(label_order)))
ax.set_xticklabels(label_order, rotation=45, ha="right")
ax.set_yticklabels(label_order)
ax.set_title("Confusion Matrix (Counts)")
for i in range(cm_array.shape[0]):
    for j in range(cm_array.shape[1]):
        ax.text(j, i, str(cm_array[i, j]), ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(REPORT_DIR / "phase5_unseen_confusion_counts.png", dpi=140)
plt.close(fig)

row_sums = cm_array.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm_array, row_sums, out=np.zeros_like(cm_array, dtype=float), where=row_sums != 0)
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_norm, cmap="Greens", vmin=0.0, vmax=1.0)
ax.set_xticks(range(len(label_order)))
ax.set_yticks(range(len(label_order)))
ax.set_xticklabels(label_order, rotation=45, ha="right")
ax.set_yticklabels(label_order)
ax.set_title("Confusion Matrix (Row Normalized)")
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        ax.text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", fontsize=9)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
fig.tight_layout()
fig.savefig(REPORT_DIR / "phase5_unseen_confusion_normalized.png", dpi=140)
plt.close(fig)

source_rows = []
for source, g in pred_df.groupby("source"):
    yt = g["y_true"].map(label2id).to_numpy()
    yp = g["y_pred"].map(label2id).to_numpy()
    source_rows.append({
        "source": source,
        "rows": int(len(g)),
        "accuracy": float(accuracy_score(yt, yp)),
        "macro_f1": float(f1_score(yt, yp, average="macro", zero_division=0)),
    })
source_metrics_df = pd.DataFrame(source_rows).sort_values("rows", ascending=False)
source_metrics_df.to_csv(REPORT_DIR / "phase5_unseen_source_metrics.csv", index=False, encoding="utf-8")
ax = source_metrics_df.set_index("source")[["accuracy", "macro_f1"]].plot(kind="bar", figsize=(8, 4), ylim=(0, 1), title="Per-Source Metrics")
ax.set_ylabel("Score")
ax.figure.tight_layout()
ax.figure.savefig(REPORT_DIR / "phase5_unseen_source_metrics.png", dpi=140)
plt.close(ax.figure)

if not np.isnan(pred_conf).all():
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(pred_df.loc[pred_df["is_correct"], "pred_confidence"], bins=30, alpha=0.6, label="correct")
    ax.hist(pred_df.loc[~pred_df["is_correct"], "pred_confidence"], bins=30, alpha=0.6, label="incorrect")
    ax.set_title("Prediction Confidence Distribution")
    ax.set_xlabel("Confidence")
    ax.set_ylabel("Rows")
    ax.legend()
    fig.tight_layout()
    fig.savefig(REPORT_DIR / "phase5_unseen_confidence_hist.png", dpi=140)
    plt.close(fig)

source_label_ct = pd.crosstab(pred_df["source"], pred_df["y_true"]).reindex(columns=label_order, fill_value=0)
ax = source_label_ct.plot(kind="bar", stacked=True, figsize=(9, 4), title="Unseen True Label Mix by Source")
ax.set_xlabel("Source")
ax.set_ylabel("Rows")
ax.figure.tight_layout()
ax.figure.savefig(REPORT_DIR / "phase5_unseen_source_label_mix.png", dpi=140)
plt.close(ax.figure)

if int(true_counts.get("DISINFO", 0)) < 100:
    log("WARNING: DISINFO support is low in unseen set; DISINFO metrics may be unstable.")

log(f"Analytics complete. Artifacts saved to: {REPORT_DIR}")
source_metrics_df


2026-03-16 00:40:19 | Encoding start | rows=57130 | chunk_size=2000 | batch_size=32
2026-03-16 00:40:19 | Encoding chunk 1-2000/57130
2026-03-16 00:43:23 | Progress 2000/57130 | rate=10.9 rows/s | eta=84.6 min
2026-03-16 00:43:23 | Encoding chunk 2001-4000/57130
2026-03-16 00:46:04 | Progress 4000/57130 | rate=11.6 rows/s | eta=76.4 min
2026-03-16 00:46:04 | Encoding chunk 4001-6000/57130
2026-03-16 00:49:32 | Progress 6000/57130 | rate=10.9 rows/s | eta=78.5 min
2026-03-16 00:49:32 | Encoding chunk 6001-8000/57130
2026-03-16 00:52:19 | Progress 8000/57130 | rate=11.1 rows/s | eta=73.7 min
2026-03-16 00:52:19 | Encoding chunk 8001-10000/57130
2026-03-16 00:55:04 | Progress 10000/57130 | rate=11.3 rows/s | eta=69.5 min
2026-03-16 00:55:04 | Encoding chunk 10001-12000/57130
2026-03-16 00:57:46 | Progress 12000/57130 | rate=11.5 rows/s | eta=65.6 min
2026-03-16 00:57:46 | Encoding chunk 12001-14000/57130
2026-03-16 01:01:02 | Progress 14000/57130 | rate=11.3 rows/s | eta=63.8 min
2026-03-

,source,rows,accuracy,macro_f1
1,gossip_lanka,33337,0.625851,0.435625
2,youtube,12462,0.763120,0.465677
0,elakiri,11331,0.699320,0.462222
